In [2]:
import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# --------------------------------------------------
# Function 8 - Week 8 Bayesian Optimisation
# --------------------------------------------------
#
# - maximise raw objective
# - use all observations through Week 7
# - ARD Matern GP
# - automatic hyperparameter optimisation
# - larger candidate pool because Function 8 is 8D
# - EI primary
# - UCB used to inspect exploration/exploitation

In [3]:
X = np.load("function8/initial_inputs.npy")
Y = np.load("function8/initial_outputs.npy").reshape(-1)

assert len(X) == len(Y)
assert X.shape[1] == 8

best_idx = np.argmax(Y)
best_x = X[best_idx]
best_y = Y[best_idx]

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nCurrent best observed input:")
print(best_x)

print("\nCurrent best observed output:")
print(best_y)

print("\nY range:")
print("min =", np.min(Y))
print("max =", np.max(Y))
print("std =", np.std(Y))

X shape: (47, 8)
Y shape: (47,)

Current best observed input:
[0.095473 0.144081 0.145356 0.087993 0.834791 0.549036 0.181794 0.551605]

Current best observed output:
9.991464829113

Y range:
min = 5.5921933895401965
max = 9.991464829113
std = 1.1534575579792214


In [4]:
kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(1e-3, 1e3)
    )
    *
    Matern(
        length_scale=np.ones(8) * 0.2,
        length_scale_bounds=(0.01, 2.0),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(1e-8, 1e-1)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=20,
    random_state=42
)

gp.fit(X, Y)

print("\nFitted kernel:")
print(gp.kernel_)


Fitted kernel:
0.914**2 * Matern(length_scale=[1.25, 2, 0.98, 2, 2, 2, 1.35, 2], nu=2.5) + WhiteKernel(noise_level=1e-08)


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/co

In [5]:
lengthscales = gp.kernel_.k1.k2.length_scale

inverse_ls = 1.0 / lengthscales
relative_sensitivity = inverse_ls / inverse_ls.sum()

print("\nARD lengthscales:")
print(lengthscales)

print("\nNormalised inverse-lengthscale sensitivity:")
print(relative_sensitivity)


ARD lengthscales:
[1.25185432 2.         0.98027299 2.         2.         2.
 1.35197213 2.        ]

Normalised inverse-lengthscale sensitivity:
[0.15791229 0.09884159 0.20166136 0.09884159 0.09884159 0.09884159
 0.14621839 0.09884159]


In [6]:
local_scale = np.clip(
    0.25 * lengthscales,
    0.015,
    0.10
)

wide_scale = np.clip(
    0.50 * lengthscales,
    0.04,
    0.20
)

print("\nLocal widths:", local_scale)
print("Wide widths:", wide_scale)


Local widths: [0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1]
Wide widths: [0.2 0.2 0.2 0.2 0.2 0.2 0.2 0.2]


In [7]:
rng = np.random.default_rng(42)

local_candidates = (
    best_x
    + rng.normal(
        0,
        local_scale,
        size=(120000, 8)
    )
)

wide_candidates = (
    best_x
    + rng.normal(
        0,
        wide_scale,
        size=(80000, 8)
    )
)

global_candidates = rng.uniform(
    0,
    1,
    size=(250000, 8)
)

local_candidates = np.clip(
    local_candidates,
    0,
    1
)

wide_candidates = np.clip(
    wide_candidates,
    0,
    1
)

candidates = np.vstack([
    local_candidates,
    wide_candidates,
    global_candidates
])

print(
    "Candidates before filtering:",
    len(candidates)
)

Candidates before filtering: 450000


In [8]:
tree = cKDTree(X)

distance, _ = tree.query(
    candidates,
    k=1
)

candidates = candidates[
    distance > 0.01
]

print(
    "Candidates after filtering:",
    len(candidates)
)

Candidates after filtering: 449999


In [9]:
mu, sigma = gp.predict(
    candidates,
    return_std=True
)

print("Predictions complete.")

Predictions complete.


In [10]:
def expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
):

    improvement = mu - best_y - xi

    Z = np.zeros_like(mu)

    valid = sigma > 1e-12

    Z[valid] = (
        improvement[valid]
        / sigma[valid]
    )

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid] * norm.cdf(Z[valid])
        +
        sigma[valid] * norm.pdf(Z[valid])
    )

    return EI

In [11]:
EI = expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
)

ei_idx = np.argmax(EI)

print("\nPRIMARY EI RESULT")

print("candidate =", candidates[ei_idx])
print("mean =", mu[ei_idx])
print("std =", sigma[ei_idx])
print("EI =", EI[ei_idx])


PRIMARY EI RESULT
candidate = [0.         0.00149804 0.         0.24678861 0.71150188 0.87034999
 0.         0.58793621]
mean = 9.858503893991438
std = 0.23008998443085485
EI = 0.04022555840441479


In [12]:
y_scale = np.std(Y)

xi_values = [
    0.0,
    0.01 * y_scale,
    0.05 * y_scale,
    0.10 * y_scale
]

print("\nEI sensitivity check:\n")

for xi in xi_values:

    EI_test = expected_improvement(
        mu,
        sigma,
        best_y,
        xi
    )

    idx = np.argmax(EI_test)

    print(
        "xi =", f"{xi:.6e}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n EI =", round(EI_test[idx], 8),
        "\n"
    )


EI sensitivity check:

xi = 0.000000e+00 
 candidate = [0.         0.00149804 0.         0.24678861 0.71150188 0.87034999
 0.         0.58793621] 
 mean = 9.858504 
 std = 0.23009 
 EI = 0.04022556 

xi = 1.153458e-02 
 candidate = [0.         0.00149804 0.         0.24678861 0.71150188 0.87034999
 0.         0.58793621] 
 mean = 9.858504 
 std = 0.23009 
 EI = 0.03707318 

xi = 5.767288e-02 
 candidate = [0.         0.         0.08204345 0.         0.79347654 0.98022354
 0.         0.78174602] 
 mean = 9.834051 
 std = 0.249056 
 EI = 0.02672571 

xi = 1.153458e-01 
 candidate = [0.         0.00306336 0.         0.15988562 0.67416891 1.
 0.         0.74496081] 
 mean = 9.808515 
 std = 0.266285 
 EI = 0.01755393 



In [13]:
print("\nUCB diagnostic:\n")

for beta in [0.1, 0.25, 0.5, 1.0]:

    UCB = mu + beta * sigma
    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n UCB =", round(UCB[idx], 6),
        "\n"
    )


UCB diagnostic:

beta=0.1 
 candidate = [0.09040729 0.18317699 0.12110555 0.12821883 0.77823005 0.57926836
 0.15775406 0.57259837] 
 mean = 9.99751 
 std = 0.020701 
 UCB = 9.99958 

beta=0.25 
 candidate = [0.10748302 0.10246352 0.11540771 0.18193642 0.81284469 0.6108236
 0.14921134 0.60689407] 
 mean = 9.992824 
 std = 0.047989 
 UCB = 10.004821 

beta=0.5 
 candidate = [0.06013371 0.08280073 0.08039653 0.20283691 0.73938424 0.65213583
 0.14000698 0.57105044] 
 mean = 9.978952 
 std = 0.085116 
 UCB = 10.021509 

beta=1.0 
 candidate = [0.03932677 0.         0.05753331 0.16465487 0.73896957 0.8335747
 0.         0.61367899] 
 mean = 9.899414 
 std = 0.192473 
 UCB = 10.091888 



In [14]:
mean_idx = np.argmax(mu)

print("\nHighest predicted mean:")
print("candidate =", candidates[mean_idx])
print("mean =", mu[mean_idx])
print("std =", sigma[mean_idx])


Highest predicted mean:
candidate = [0.10216723 0.16474697 0.12789528 0.11822681 0.80849631 0.56298789
 0.14983183 0.60891298]
mean = 9.997853429176788
std = 0.017162697344203445


In [15]:
# --------------------------------------------------
# Final Function 8 Week 8 selection
# --------------------------------------------------
#
# EI is strongly uncertainty-driven and moves toward
# several domain boundaries with a much lower GP mean.
#
# The highest-mean and low-beta UCB candidates remain
# in the established high-performing region.
#
# beta = 0.25 gives a useful increase in uncertainty
# relative to pure exploitation while retaining a
# predicted mean close to the current best.
#
# This is preferred in 8D, where aggressive
# uncertainty-seeking is particularly risky.

beta = 0.25

UCB = mu + beta * sigma
final_idx = np.argmax(UCB)

week8_candidate = candidates[final_idx]

print("Week 8 Function 8 candidate:")
print(week8_candidate)

print("\nPredicted mean:")
print(mu[final_idx])

print("\nPredicted std:")
print(sigma[final_idx])

print("\nUCB:")
print(UCB[final_idx])

portal = "-".join(
    f"{x:.6f}"
    for x in week8_candidate
)

print("\nPortal format:")
print(portal)

Week 8 Function 8 candidate:
[0.10748302 0.10246352 0.11540771 0.18193642 0.81284469 0.6108236
 0.14921134 0.60689407]

Predicted mean:
9.992823677241477

Predicted std:
0.047988911625425526

UCB:
10.004820905147833

Portal format:
0.107483-0.102464-0.115408-0.181936-0.812845-0.610824-0.149211-0.606894
